# Macedonian OCR — PDF scanner (Colab)

OCR for Macedonian Cyrillic documents: PaddleOCR with the custom-trained [mjurukov/macedonian-ocr-v8](https://huggingface.co/mjurukov/macedonian-ocr-v8) recognition model and PP-OCRv6 detection.

**How to use:**
1. Menu *Runtime → Change runtime type* → select **T4 GPU**.
2. Menu *Runtime → Run all*.
3. When the upload dialog appears (cell 4), select one or more PDF files.
4. When processing finishes, `results.zip` downloads automatically.

The zip contains, per PDF:
- `<name>.txt` — recognized text, page-separated
- `<name>/page_NNN.jpg` — per-page image with detected text boxes

The model downloads automatically from Hugging Face — nothing to set up.

In [ ]:
# 1. Install dependencies (~3-4 min)
%pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
%pip install -q paddleocr==3.7.0 pymupdf

# paddle bundles its own CUDA libs which downgrade Colab's torch-compatible
# versions — reinstall the exact versions torch requires
import importlib.metadata, subprocess
for pkg in ('nvidia-nccl-cu12', 'nvidia-cudnn-cu12', 'nvidia-cusparselt-cu12'):
    try:
        ver = next(r.split(';')[0].strip()
                   for r in importlib.metadata.requires('torch')
                   if r.startswith(pkg))
    except StopIteration:
        continue
    subprocess.run(['pip', 'install', '-q', ver], check=False)

import paddle
print('Paddle device:', paddle.device.get_device())  # should say gpu:0

In [ ]:
# 2. Download the V8 Macedonian recognition model from Hugging Face
import os
from huggingface_hub import snapshot_download

REC_DIR = snapshot_download('mjurukov/macedonian-ocr-v8',
                            local_dir='/content/mk_rec_infer')
print(os.listdir(REC_DIR))  # expect inference.json / .pdiparams / .yml

In [ ]:
# 3. Load PaddleOCR: PP-OCRv6 medium detector + Macedonian V8 recognizer
os.environ['PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK'] = 'True'

from paddleocr import PaddleOCR

ocr = PaddleOCR(
    text_detection_model_name='PP-OCRv6_medium_det',
    text_recognition_model_name='PP-OCRv5_server_rec',
    text_recognition_model_dir=REC_DIR,
    text_det_thresh=0.15,
    text_det_box_thresh=0.28,
    text_det_unclip_ratio=3.0,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
)

In [ ]:
# 4. Upload PDF files to scan
from google.colab import files

SRC_DIR = '/content/pdfs'
os.makedirs(SRC_DIR, exist_ok=True)

uploaded = files.upload()
for name in uploaded:
    os.rename(name, os.path.join(SRC_DIR, name))
print(f'{len(uploaded)} PDF(s) ready.')

In [ ]:
# 5. OCR every uploaded PDF, then download results.zip
import glob
import shutil
import cv2
import fitz
import numpy as np

OUT_DIR = '/content/output'
DPI = 200
SCORE_MIN = 0.3


def render_page(page):
    zoom = DPI / 72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), colorspace=fitz.csRGB)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    return cv2.cvtColor(img, cv2.COLOR_RGB2BGR)


def draw_boxes(img, polys, scores):
    vis = img.copy()
    for poly, score in zip(polys, scores):
        if score < SCORE_MIN:
            continue
        pts = np.array(poly, dtype=np.int32).reshape(-1, 1, 2)
        color = (80, 200, 0) if score >= 0.7 else (0, 160, 255)  # BGR
        cv2.polylines(vis, [pts], isClosed=True, color=color, thickness=2)
    return vis


pdfs = sorted(glob.glob(os.path.join(SRC_DIR, '*.pdf')))
if not pdfs:
    raise SystemExit('No PDFs uploaded — run the previous cell first.')

os.makedirs(OUT_DIR, exist_ok=True)

for n, pdf_path in enumerate(pdfs, 1):
    stem = os.path.splitext(os.path.basename(pdf_path))[0]
    page_dir = os.path.join(OUT_DIR, stem)
    txt_path = os.path.join(OUT_DIR, stem + '.txt')
    if os.path.exists(txt_path):
        print(f'[{n}/{len(pdfs)}] SKIP (done) {stem}', flush=True)
        continue
    os.makedirs(page_dir, exist_ok=True)

    doc = fitz.open(pdf_path)
    all_text = []
    for i, page in enumerate(doc, 1):
        img = render_page(page)
        result = ocr.predict(img)

        lines, polys, scores = [], [], []
        for res in result:
            polys.extend(res.get('rec_polys', []))
            scores.extend(res.get('rec_scores', []))
            for t, s in zip(res.get('rec_texts', []), res.get('rec_scores', [])):
                if s >= SCORE_MIN and t.strip():
                    lines.append(t)

        vis = draw_boxes(img, polys, scores)
        cv2.imwrite(os.path.join(page_dir, f'page_{i:03d}.jpg'), vis,
                    [cv2.IMWRITE_JPEG_QUALITY, 88])

        all_text.append(f'--- \u0421\u0442\u0440\u0430\u043d\u0438\u0446\u0430 {i} ---\n' + '\n'.join(lines))
        print(f'[{n}/{len(pdfs)}] {stem} p{i}/{len(doc)}: {len(lines)} lines', flush=True)

    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write('\n\n'.join(all_text) + '\n')

shutil.make_archive('/content/results', 'zip', OUT_DIR)
files.download('/content/results.zip')
print('Done.')